In [1]:
import os
import sys
import json
from datetime import datetime

os.chdir("/home/kaariaa3/mscthesis/")
sys.path.append("./src/")  # Add module directory to path

# Set environment variables before importing transformers
os.environ["HUGGINGFACE_HUB_CACHE"] = (
    "/scratch/shareddata/dldata/huggingface-hub-cache/hub"  # Load local models
)
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["HF_HOME"] = (
    f"{os.environ["WRKDIR"]}/.cache/huggingface"  # Cache in work directory
)

import transformers
import numpy as np
from datasets import load_dataset

from utils.helpers import (
    get_system_prompt,
    get_default_response,
    make_prompt,
    parse_output,
)

# Force reload changes
from utils.prompts import *

from utils.constants import (
    PIPE_MAX_NEW_TOKENS,
    MODEL_TEMPERATURE,
    BATCH_SIZE,
    PIPE_RETURN_FULL_TEXT
)

from utils.plots import calculate_accuracy

In [2]:
LABELS = ["yes", "no"]  # Labels per column
EVAL_COLS = [
    "The exercise description matched the selected theme (Yes/No)",
    "The exercise description matched the selected topic (Yes/No)",
    "Included concepts that were too advanced (Yes/No)",
]
PRED_COLS = ["augmentedProblemDescription", "augmentedExampleSolution"]

DEFAULT_DATA = "./data/final_dataset.csv"
DEFAULT_MODEL = "Qwen/Qwen2.5-14B-Instruct"

In [4]:
task = "detect"
n_rows = -1

use_fixed_demos = True
fixed_demos_idx = [273, 20, 0, 8, 79]

dataset = load_dataset("csv", data_files=DEFAULT_DATA, split="train", sep=";")
dataset = dataset.shuffle(seed=42)
if n_rows is not None and n_rows > 0:
    dataset = dataset.select(range(n_rows))

# Exclude rows that are used as fixed demos
if use_fixed_demos:
    dataset = dataset.select(
        (
            i for i in range(len(dataset)) 
            if i not in set(fixed_demos_idx)
        )
    )

demonstrations_rng = np.random.default_rng(seed=10)
n_demonstrations = 0

# Make user and system prompts
dataset = dataset.map(
    lambda row: {
        "user_prompt": make_prompt(row, task),
        "system_prompt": get_system_prompt(
            task,
            dataset.select(
                demonstrations_rng.integers(
                    low=0, high=dataset.num_rows, size=n_demonstrations
                )
            ),
            use_fixed_demos
        ),
    }
)

Map:   0%|          | 0/530 [00:00<?, ? examples/s]

In [6]:
for row in dataset:
    print(row["system_prompt"])
    break

Use the demonstrations below as examples on how to answer the question.

Demonstration:

Theme: literature
Topic: Agatha Christie
Concept: conditional statements

--- PROBLEM DESCRIPTION ---
Vincent van Gogh, the renowned post-impressionist painter, has a unique way of describing his paintings based on their emotional impact. The emotional impact is represented as numbers and is accompanied by the following textual descriptions:
<table>
<tr>
<th>Impact</th>
<th>Description</th>
</tr>
<tr>
<th>5</th>
<th>Masterpiece</th>
</tr>
<tr>
<th>4</th>
<th>Stunning</th>
</tr>
<tr>
<th>3</th>
<th>Impressive</th>
</tr>
<tr>
<th>2</th>
<th>Noticeable</th>
</tr>
<tr>
<th>1</th>
<th>Faint</th>
</tr>
</table>
Write a program that asks the user for a number and prints the textual description related to that number. If the user enters any other number, the program should print the message <code>Invalid impact!</code>.

Below is an example of the expected operation of the program.

<pre>
What impact?
<b>&

In [5]:
params = {
    "model": DEFAULT_MODEL,
    "device_map": 0,  # Force GPU
    "max_new_tokens": PIPE_MAX_NEW_TOKENS,
    "temperature": MODEL_TEMPERATURE,
}

pipeline = transformers.pipeline("text-generation", **params)
pipeline.tokenizer.pad_token = pipeline.tokenizer.eos_token
pipeline.model.config.pad_token_id = pipeline.model.config.eos_token_id

2026-03-20 11:24:45.667290: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

Device set to use cuda:0


In [6]:
s = datetime.now()

default_response = get_default_response(task)
results = {key: [] for key in default_response.keys()}

batch_size = BATCH_SIZE
prompts = dataset["user_prompt"]
system_prompts = dataset["system_prompt"]
for i in range(0, len(prompts), batch_size):

    # Build batched conversation inputs
    batch_prompts = zip(system_prompts[i : i + batch_size], prompts[i : i + batch_size])
    batch_inputs = [
        [
            {"role": "system", "content": sp},
            {"role": "user", "content": up},
        ]
        for sp, up in batch_prompts
    ]

    # Single batched forward pass
    outputs = pipeline(
        batch_inputs,
        return_full_text=PIPE_RETURN_FULL_TEXT,
    )

    # Process each output in the batch
    for output in outputs:
        text = output[0]["generated_text"]
        parsed = parse_output(text)

        for key, value in default_response.items():
            results[key].append(json.dumps(parsed.get(key, value)))

e = datetime.now()

print(f"Done. Elapsed time: {e - s}")

Done. Elapsed time: 0:01:27.793733


In [7]:
def print_row(row):
    print("##################")
    print("#  Ground-truth  #")
    print("##################")
    print("Theme correct: " + row[EVAL_COLS[0]])
    print("Topic correct: " + row[EVAL_COLS[1]])
    print("Uses advanced concepts: " + row[EVAL_COLS[2]])
    print("Original concept: " + row["concept"])
    print(row["system_prompt"])
    print(row["user_prompt"])

    print("##################")
    print("#  Prediction    #")
    print("##################")
    print("Theme correct: " + results["themeCorrect"][i])
    print("Topic correct: " + results["topicCorrect"][i])
    print("Uses advanced concepts: " + results["usesAdditionalConcepts"][i])
    print("Explanation: " + results["explanation"][i])
    print("---------------------------------------------------")
    print()


print_n_rows = 10
if not True:
    for i, row in data.iloc[:print_n_rows, :].iterrows():
        print_row(row)

In [8]:
for key, value in results.items():
    dataset = dataset.add_column(key, value)

In [9]:
calculate_accuracy(dataset.to_pandas())

{'theme_accuracy': np.float64(0.8),
 'topic_accuracy': np.float64(0.9),
 'concept_accuracy': np.float64(0.2),
 'total_accuracy': np.float64(0.1)}